In [ ]:
# Kaggle-only setup
import os
from pathlib import Path

# Find the project root inside the attached dataset(s)
base = Path("/kaggle/input")
candidates = list(base.rglob("project_root.py"))
if not candidates:
    raise FileNotFoundError("project_root.py not found under /kaggle/input. Did you attach the dataset in the right sidebar?")
# Prefer the shortest path (closest match)
candidates.sort(key=lambda p: len(str(p)))
PROJECT_ROOT = candidates[0].parent

# Make relative paths (e.g., processed/02_make_windows.py) work as-is
os.chdir(PROJECT_ROOT)

from project_root import (  # noqa: E402
    ROOT,
    WORK_ROOT,
    DATA_RAW,
    PROCESSED,
    ALIGNED,
    WINDOWS,
    SPLITS,
    MODELS,
    ensure_working_tree,
)

ensure_working_tree()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ROOT       :", ROOT)
print("WORK_ROOT  :", WORK_ROOT)
print("DATA_RAW   :", DATA_RAW)
print("PROCESSED  :", PROCESSED)


In [ ]:
# Kaggle-only notebook: attach your dataset via the right sidebar (Add Data), then run from the top.


In [ ]:
# Runs all preprocessing, alignment, and saves aligned arrays
%run src/01_preprocessing.py

In [ ]:
!python src/02_make_windows.py --window 20

In [ ]:
!python src/03_train_val_split.py --window 20

In [ ]:
!python src/04_normalize_windows.py --window 20

In [ ]:
!python src/05_train_rnn.py --window 20 --batch 64 --train_fraction 0.5

In [ ]:
!python src/05_train_lstm.py --window 10

In [ ]:
!python src/06_train_transformer.py --window 10

In [ ]:
!python src/07_evaluate_models.py --window 20 --model_path processed/models/rnn_C_w20_best.keras --out_name rnn_C

In [ ]:
!python - << 'PY'
import numpy as np
from pathlib import Path

PROCESSED = Path("processed")
W = 10  # change to 20 later if needed

X = np.load(PROCESSED / f"windows_w{W}_norm/X.npy", mmap_mode="r")
y = np.load(PROCESSED / f"windows_w{W}_norm/y.npy", mmap_mode="r")
train_idx = np.load(PROCESSED / "splits" / f"splits_w{W}" / "train_idx.npy")

Xtr = np.asarray(X[train_idx], dtype=np.float32).reshape(-1, X.shape[-1])
ytr = np.asarray(y[train_idx], dtype=np.float32)

print("X train mean (avg |mean|):", float(np.mean(np.abs(Xtr.mean(axis=0)))))
print("X train std  (avg |std-1|):", float(np.mean(np.abs(Xtr.std(axis=0) - 1))))
print("y train mean:", float(np.mean(ytr.mean(axis=0))))
print("y train std :", float(np.mean(ytr.std(axis=0))))


In [ ]:
!python processed/plot_learning_curve.py --history processed/results_step08/tr_C_w10_history.json